# 데이터 로드 및 전처리 파이프라인

1. 데이터 로드
2. 기반 데이터프레임 병합
3. 전처리(불필요 칼럼 일괄 삭제)

In [1]:
import pandas as pd
import os

DATA_DIR = "data"

## 1. 데이터 로드

In [2]:
def load_data():
    files = {
        '기본정보': '기본정보.csv', '면적정보': '면적정보.csv', '시설정보': '시설정보.csv',
        '운영정보': '운영정보.csv', '위치정보': '위치정보.csv', '장기수선': '장기수선.csv', '관리비': '관리비.csv'
    }
    dfs = {}
    for name, filename in files.items():
        path = os.path.join(DATA_DIR, filename)
        try:
            dfs[name] = pd.read_csv(path, encoding='utf-8')
        except UnicodeDecodeError:
            dfs[name] = pd.read_csv(path, encoding='cp949')
        print(f"[{name}] 데이터 로드 완료: {dfs[name].shape}")
    return dfs

dfs = load_data()

[기본정보] 데이터 로드 완료: (2454, 19)
[면적정보] 데이터 로드 완료: (13852, 7)
[시설정보] 데이터 로드 완료: (2454, 12)
[운영정보] 데이터 로드 완료: (2454, 12)
[위치정보] 데이터 로드 완료: (2454, 3)
[장기수선] 데이터 로드 완료: (147240, 10)


[관리비] 데이터 로드 완료: (147240, 38)


## 2. 모든 데이터 우선 병합 (기반 데이터프레임 생성)

In [3]:
def merge_raw_data(dfs):
    if '면적정보' in dfs:
        dfs['면적정보'] = dfs['면적정보'].drop_duplicates(subset=['단지코드']).drop(columns=['주거전용면적(세부)', '세대수'], errors='ignore')

    print("\n[진행] 정적 테이블 병합 시작...")
    static_df = dfs['기본정보']
    for table_name in ['면적정보', '시설정보', '운영정보', '위치정보']:
        df_to_merge = dfs[table_name]
        cols_to_use = [col for col in df_to_merge.columns if col not in static_df.columns or col == '단지코드']
        static_df = pd.merge(static_df, df_to_merge[cols_to_use], on='단지코드', how='left')
    
    print("\n[진행] 시계열 테이블 병합 시작...")
    ts_df = dfs['관리비']
    repair_df = dfs['장기수선']
    join_keys = ['단지코드', '발생년월(YYYYMM)']
    cols_to_use = [col for col in repair_df.columns if col not in ts_df.columns or col in join_keys]
    ts_df = pd.merge(ts_df, repair_df[cols_to_use], on=join_keys, how='left')
    
    print("\n[진행] 최종 전체 원본 병합 시작...")
    raw_master_df = pd.merge(ts_df, static_df, on='단지코드', how='left')
    print(f"기반이 되는 원본 Master 데이터 병합 완료: {raw_master_df.shape}")
    return raw_master_df

raw_master_df = merge_raw_data(dfs)


[진행] 정적 테이블 병합 시작...

[진행] 시계열 테이블 병합 시작...

[진행] 최종 전체 원본 병합 시작...


기반이 되는 원본 Master 데이터 병합 완료: (147240, 90)


## 3. 전처리: 병합된 프레임에서 불필요 칼럼 일괄 삭제

In [4]:
def drop_unnecessary_columns(df):
    cols_to_drop = [
        '차량유지비', '지능형네트워크유지비', '재해예방비', '가스사용료(공용)', '가스사용료(전용)', '기타', '제세공과금', '교육훈련비', '시설유지비', '안전점검비', '위탁관리수수료', '급탕비(공용)', '수도료(공용)', 'TV수신료', '정화조오물수수료', '선관위운영비',
        '시공사', '주택관리업자',
        '건물구조', '전기-수전용량', '전기-세대전기계약방식', '승강기관리-관리방식', 'CCTV대수', '부대복리시설', '홈네트워크',
        '경비관리-계약업체', '청소관리-계약업체', '음식물 처리방법', '소독관리-계약업체', '일반관리-관리방식', '경비관리-관리방식', '청소관리-관리방식', '소독관리-관리방식',
        '입주자기여수익', '공동기여수익'
    ]
    print("\n[진행] 병합된 데이터에서 전처리 시작...")
    existing_cols = [col for col in cols_to_drop if col in df.columns]
    df_cleaned = df.drop(columns=existing_cols)
    print(f"삭제된 칼럼 수: {len(existing_cols)}개")
    print(f"전처리 완료 후 최종 남은 데이터 크기: {df_cleaned.shape}")
    return df_cleaned

final_df = drop_unnecessary_columns(raw_master_df)


[진행] 병합된 데이터에서 전처리 시작...
삭제된 칼럼 수: 35개
전처리 완료 후 최종 남은 데이터 크기: (147240, 55)


## 4. 최종 결과 확인

In [5]:
final_df.head()

,단지코드,발생년월(YYYYMM),공용관리비계,인건비,제사무비,피복비,그밖의부대비용,청소비,경비비,소독비,...,주거전용면적(단지합계),승강기대수,총주차대수,최고층수,지하층수,일반관리-인원,경비관리-인원,청소관리-인원,위도,경도
0,A11007001,202101,53343583.0,21552672,248500.0,0,341860,9046000,18157174,250000,...,22444.8,5,315,16,3,5,4.0,4,37.573567,126.972151
1,A11007001,202102,53189158.0,21552614,248100.0,0,326160,9046000,18157174,250000,...,22444.8,5,315,16,3,5,4.0,4,37.573567,126.972151
2,A11007001,202103,53228835.0,21552616,228500.0,0,711840,9046000,18157174,250000,...,22444.8,5,315,16,3,5,4.0,4,37.573567,126.972151
3,A11007001,202104,51995807.0,20485440,243900.0,0,315660,9046000,18157174,250000,...,22444.8,5,315,16,3,5,4.0,4,37.573567,126.972151
4,A11007001,202105,52384391.0,20494660,231100.0,0,398410,9046000,18157174,250000,...,22444.8,5,315,16,3,5,4.0,4,37.573567,126.972151


In [6]:
print(list(final_df.columns))

['단지코드', '발생년월(YYYYMM)', '공용관리비계', '인건비', '제사무비', '피복비', '그밖의부대비용', '청소비', '경비비', '소독비', '승강기유지비', '수선비', '개별사용료계', '난방비(공용)', '난방비(전용)', '급탕비(전용)', '전기료(공용)', '전기료(전용)', '수도료(전용)', '생활폐기물수수료', '입대의운영비', '건물보험료', '시도_x', '장충금 월부과액', '장충금 월사용액', '장충금 총적립금액', '장충금 적립률', '잡수입 월수입금액', '시도_y', '시군구', '동리', '단지명', '단지분류', '법정동주소', '도로명주소', '분양형태', '사용승인일', '동수', '세대수', '분양세대수', '임대세대수', '관리방식', '난방방식', '복도유형', '관리비부과면적', '주거전용면적(단지합계)', '승강기대수', '총주차대수', '최고층수', '지하층수', '일반관리-인원', '경비관리-인원', '청소관리-인원', '위도', '경도']
